In [1]:
import sys
import json
import pandas as pd

sys.path.append("..")

from etl_pipeline_local import get_btts_table

In [2]:
# Data upload - do not implement
# Bet setup
target_col = "btts"

# Load data
df_loaded = get_btts_table()
df_loaded = df_loaded.sort_values('time').reset_index(drop=True)
df = df_loaded.copy()

In [3]:
def get_mask(df, params_dict):
    mask = pd.Series(True, index=df.index)
    features = set()

    for key in params_dict:
        for suffix in ["_cat", "_use_min", "_use_max", "_min", "_max", "_include_missing"]: # ,   "_min_idx", "_max_idx", 
             if key.endswith(suffix):
                features.add(key[:-len(suffix)])
                break

    for feat in features:
        s = df[feat]

        include_missing = params_dict.get(f"{feat}_include_missing", False)
        use_min = params_dict.get(f"{feat}_use_min", True)
        use_max = params_dict.get(f"{feat}_use_max", True)

        feat_mask = pd.Series(True, index=df.index)

        # Categorical filtering
        if f"{feat}_cat" in params_dict:
            feat_mask &= s.isin(params_dict[f"{feat}_cat"])

        # Numerical filtering
        if f"{feat}_min" in params_dict and use_min:
            feat_mask &= s >= params_dict[f"{feat}_min"]

        if f"{feat}_max" in params_dict and use_max:
            feat_mask &= s <= params_dict[f"{feat}_max"]

        if include_missing:
            feat_mask = feat_mask | s.isna()
        else:
            feat_mask = feat_mask & s.notna()

        if params_dict[f"use_{feat}"]:
            mask &= feat_mask

    return mask


def round_to_step(x, step):
    """
    Arrotonda x a un multiplo di 'step'.
    """
    try:
        if not step:
            return str(x)
        elif step <= 0:
            raise ValueError("step deve essere > 0")
        else:
            ratio = x / step
            return round(ratio) * step
    
    except Exception as e: 
        return None

In [ ]:
with open("../../strategies/strategies.json", "r", encoding="utf-8") as f:
    data = json.load(f)

params_dict = data[target_col]['params_dict']
feature_bins_map = data[target_col]['feature_bins_map']

df_binned = df.copy()

for feat, step in feature_bins_map.items():
    df[feat] = [round_to_step(x, step) for x in df[feat]]

    if isinstance(step, int):
        df[feat] = df[feat].astype("Int64")


mask = get_mask(df, params_dict)

df_filtered = df[mask]
df_filtered.head()

,time,chance1x2_chance_p1,chance1x2_chance_px,chance1x2_chance_p2,chance1x2_chance_p1x,chance1x2_chance_p2x,chance1x2_chance_p12,chance1x2_chance_pHt1,chance1x2_chance_pHtx,chance1x2_chance_pHt2,...,underOver_flashback_under35,underOver_flashback_over35,team_goal,team_goalHt,team_corner,chance1x2_xg_home,chance1x2_xg_away,chance1x2_xg_total,chance1x2_xg,btts
5,2025-05-16 18:00:00,64.1,20.7,15.2,84.8,35.9,79.3,42.5,40.3,17.2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True
8,2025-05-16 18:00:00,51.9,23.8,24.3,75.7,48.1,76.2,39.7,40.6,19.7,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True
84,2025-05-17 14:00:00,48.0,24.2,27.8,72.2,52.0,75.8,33.8,41.6,24.6,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True
90,2025-05-17 15:30:00,73.8,15.9,10.3,89.7,26.2,84.1,55.8,24.7,19.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
108,2025-05-17 18:00:00,30.3,26.5,43.2,56.8,69.7,73.5,24.2,41.5,34.3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False


In [5]:
df_filtered.shape

(107, 139)